# CFR & CAFD-LC on Wikipedia KB (KB Confound Fix)

**Goal**: Run CFR and CAFD-LC using the same Wikipedia KB as Li et al.  
This closes the KB confound issue: currently SDCP vs CFR/CAFD-LC compares different KBs.

- KB: `articles_l3.pkl`, 999 articles, chunk_size=64, overlap=8 (same as Li et al.)
- Datasets: TruthfulQA (615Q) + MMLU (1596Q)
- Methods: CFR, CAFD-LC
- Expected runtime: ~2.5 hours (CFR TQA ~21min, CFR MMLU ~55min, CAFD-LC similar)

**Note**: Wikipedia KB articles are plain text — no `correct_answers`/`incorrect_answers` fields.  
CFR and CAFD-LC must retrieve their contrastive labels from the **dataset-derived KB**  
(same as before), but use Wikipedia articles as the **retrieval corpus** for context sentences.

In [1]:
# ── Cell 0: Setup ─────────────────────────────────────────────────────────────
import os, sys, gc, re, time, json
import numpy as np, pandas as pd
import torch, faiss
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer as rs_module
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime
from tqdm import tqdm

BASE_DIR     = '/home/user/RAG_Best_Practices/RAG_best_practices-main'
DATASETS_DIR = '/home/user/RAG_Best_Practices/datasets'
OUT_DIR      = '/home/user/RAG_Best_Practices'
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID     = 'mistralai/Mistral-7B-Instruct-v0.2'

INST_S = '[INST]'
INST_E = '[/INST]'
SYS    = ('You are a truthful expert question-answering bot and should '
           'correctly and concisely answer the following question')

CFR_PARAMS  = {'alpha': 0.5, 'beta': 0.3, 'gamma': 0.2, 'focus': 20}
CAFD_PARAMS = {'analysis_max_tokens': 60, 'max_new_tokens': 25}

print(f'Device: {DEVICE}')
print('Loading Mistral-7B (4-bit NF4)...')
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_cfg, device_map='auto'
)
llm.eval()
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded ✓')

Device: cuda
Loading Mistral-7B (4-bit NF4)...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded ✓


In [2]:
# ── Cell 1: Load datasets — EXACT same splits as Full_Dataset_4bit.ipynb ──────
N_TRUTHFULQA     = 615
MMLU_PER_SUBJECT = 28
RANDOM_SEED      = 42
CHOICE_LABELS    = ['A', 'B', 'C', 'D']

# ── TruthfulQA ────────────────────────────────────────────────────────────────
print('Loading TruthfulQA...')
tqa_raw = load_from_disk(f'{DATASETS_DIR}/truthfulqa').to_pandas()
tqa_all = tqa_raw[['question','best_answer','correct_answers','incorrect_answers']].copy()
tqa_all['correct_answers']   = tqa_all['correct_answers'].apply(
    lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['incorrect_answers'] = tqa_all['incorrect_answers'].apply(
    lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['best_answer']       = tqa_all['best_answer'].apply(
    lambda x: [x] if x else [])
tqa_all = tqa_all[(tqa_all['correct_answers'].apply(len) > 1) &
                   (tqa_all['incorrect_answers'].apply(len) > 1)].reset_index(drop=True)
tqa_test_idx = tqa_all.sample(n=N_TRUTHFULQA, random_state=RANDOM_SEED).index
tqa    = tqa_all.loc[tqa_test_idx].reset_index(drop=True)
tqa_kb = tqa_all.drop(tqa_test_idx).reset_index(drop=True)
del tqa_raw; gc.collect()
print(f'  TQA: test={len(tqa)}Q | kb={len(tqa_kb)}Q')

# ── MMLU ──────────────────────────────────────────────────────────────────────
print('Loading MMLU...')
mmlu_raw = load_from_disk(f'{DATASETS_DIR}/mmlu').to_pandas()

def mmlu_to_unified(row):
    choices   = list(row['choices'])
    ans_idx   = int(row['answer'])
    correct   = choices[ans_idx]
    incorrect = [choices[i] for i in range(len(choices)) if i != ans_idx]
    formatted_q = (row['question'] + '\n' +
                   '\n'.join(f'{CHOICE_LABELS[i]}) {choices[i]}' for i in range(len(choices))))
    return pd.Series({'question': formatted_q, 'question_plain': row['question'],
                      'subject': row['subject'], 'best_answer': [correct],
                      'correct_answers': [correct], 'incorrect_answers': incorrect,
                      'answer_idx': ans_idx, 'choices': choices})

mmlu_test_parts, mmlu_kb_parts = [], []
for subject, group in mmlu_raw.groupby('subject'):
    group = group.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    n_test = min(MMLU_PER_SUBJECT, len(group))
    mmlu_test_parts.append(group.iloc[:n_test])
    if len(group) > n_test:
        mmlu_kb_parts.append(group.iloc[n_test:])
mmlu    = pd.concat(mmlu_test_parts).reset_index(drop=True).apply(mmlu_to_unified, axis=1)
mmlu_kb = pd.concat(mmlu_kb_parts).reset_index(drop=True).apply(mmlu_to_unified, axis=1)
del mmlu_raw; gc.collect()
print(f'  MMLU: test={len(mmlu)}Q | kb={len(mmlu_kb)}Q')

Loading TruthfulQA...
  TQA: test=615Q | kb=100Q
Loading MMLU...
  MMLU: test=1596Q | kb=228Q


In [3]:
# ── Cell 2: Load Wikipedia KB + build FAISS index ─────────────────────────────
CHUNK_SIZE = 64
OVERLAP    = 8

print('Loading Wikipedia KB (articles_l3.pkl)...')
wiki_kb = pd.read_pickle(f'{BASE_DIR}/resources/articles_l3.pkl')
print(f'  {len(wiki_kb)} articles loaded, columns: {list(wiki_kb.columns)}')

# Chunk articles same as Li et al.
print('Chunking articles (chunk_size=64 tokens, overlap=8)...')
chunks = []
for _, row in tqdm(wiki_kb.iterrows(), total=len(wiki_kb), desc='Chunking'):
    text   = str(row.get('text_en', row.get('text', row.get('content', ''))))
    title  = str(row.get('title_en', row.get('title', '')))
    tokens = tokenizer.encode(text, add_special_tokens=False)
    for start in range(0, len(tokens), CHUNK_SIZE - OVERLAP):
        chunk_tok = tokens[start:start + CHUNK_SIZE]
        chunk_txt = tokenizer.decode(chunk_tok)
        if len(chunk_txt.strip()) > 20:
            chunks.append({'text': chunk_txt.strip(), 'title': title})

wiki_chunks = pd.DataFrame(chunks)
print(f'  {len(wiki_chunks)} chunks created')

print('Encoding Wikipedia chunks...')
wiki_embs = embed_model.encode(
    wiki_chunks['text'].tolist(), show_progress_bar=True, batch_size=256)
wiki_embs = np.array(wiki_embs, dtype=np.float32)
faiss.normalize_L2(wiki_embs)
wiki_index = faiss.IndexFlatIP(wiki_embs.shape[1])
wiki_index.add(wiki_embs)
print(f'  FAISS index: {wiki_index.ntotal} vectors ✓')

Loading Wikipedia KB (articles_l3.pkl)...
  999 articles loaded, columns: ['title_en', 'text_en', 'title_de', 'text_de', 'title_fr', 'text_fr', 'description_en', 'aliases_en', 'description_de', 'aliases_de', 'description_fr', 'aliases_fr']
Chunking articles (chunk_size=64 tokens, overlap=8)...


Chunking: 100%|██████████| 999/999 [01:32<00:00, 10.86it/s]


  213226 chunks created
Encoding Wikipedia chunks...


Batches:   0%|          | 0/833 [00:00<?, ?it/s]

  FAISS index: 213226 vectors ✓


In [4]:
# ── Cell 3: Core utilities ────────────────────────────────────────────────────
scorer = rs_module.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

def generate(prompts, max_new_tokens=25, num_beams=2):
    enc = tokenizer(prompts, return_tensors='pt', padding=True,
                    truncation=True, max_length=2048).to(DEVICE)
    input_len = enc['input_ids'].shape[1]
    with torch.no_grad():
        out = llm.generate(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            pad_token_id=tokenizer.eos_token_id
        )
    return [tokenizer.decode(r[input_len:], skip_special_tokens=True).strip()
            or 'I have no comment' for r in out]

def clean_response(text):
    for stop in ['\n\n', '\nQuestion:', '\nQ:', '---']:
        if stop in text:
            text = text[:text.index(stop)]
    return text.strip()

def retrieve_wiki(query, k=20):
    """Retrieve top-k chunks from Wikipedia index."""
    q_emb = np.array(embed_model.encode([query], show_progress_bar=False), dtype=np.float32)
    faiss.normalize_L2(q_emb)
    _, idxs = wiki_index.search(q_emb, k)
    return [wiki_chunks.iloc[i]['text'] for i in idxs[0] if i < len(wiki_chunks)]

def build_kb_index(kb_data):
    embs = embed_model.encode(kb_data['question'].tolist(),
                               show_progress_bar=True, batch_size=64)
    embs = np.array(embs, dtype=np.float32)
    faiss.normalize_L2(embs)
    idx = faiss.IndexFlatIP(embs.shape[1])
    idx.add(embs)
    return idx

def retrieve_from_kb(query, kb_idx, kb_data, k=1):
    q_emb = np.array(embed_model.encode([query], show_progress_bar=False), dtype=np.float32)
    faiss.normalize_L2(q_emb)
    _, idxs = kb_idx.search(q_emb, k)
    return [kb_data.iloc[i] for i in idxs[0] if i < len(kb_data)]

def compute_metrics(generated, references, test_data):
    r1s, r2s, rls, accs = [], [], [], []
    for gen, refs in zip(generated, references):
        refs_str = refs if isinstance(refs, list) else [refs]
        best_r1 = max(scorer.score(gen, ref)['rouge1'].fmeasure for ref in refs_str)
        best_r2 = max(scorer.score(gen, ref)['rouge2'].fmeasure for ref in refs_str)
        best_rl = max(scorer.score(gen, ref)['rougeL'].fmeasure for ref in refs_str)
        r1s.append(best_r1); r2s.append(best_r2); rls.append(best_rl)
    # Accuracy for MCQ
    if 'answer' in test_data.columns:
        for gen, row in zip(generated, test_data.itertuples()):
            choices = row.choices if isinstance(row.choices, list) else list(row.choices)
            correct = choices[int(row.answer)]
            correct_letter = chr(65 + int(row.answer))
            gen5 = gen[:50].upper()
            hit  = (correct_letter in gen5 or
                    correct.lower()[:20] in gen.lower())
            accs.append(int(hit))
    # ECS
    gen_embs = embed_model.encode(generated, show_progress_bar=False, batch_size=256)
    ref_embs = embed_model.encode(
        [r[0] if isinstance(r,list) else r for r in references],
        show_progress_bar=False, batch_size=256)
    ecs = (np.sum(gen_embs * ref_embs, axis=1) /
           (np.linalg.norm(gen_embs,axis=1) * np.linalg.norm(ref_embs,axis=1) + 1e-9))
    return {
        'R1':  round(np.mean(r1s)*100, 2),
        'R2':  round(np.mean(r2s)*100, 2),
        'RL':  round(np.mean(rls)*100, 2),
        'ECS': round(np.mean(ecs)*100, 2),
        'Acc': round(np.mean(accs)*100, 1) if accs else None,
    }

print('Utilities ready ✓')

Utilities ready ✓


In [5]:
# ── Cell 4: CFR on TruthfulQA with Wikipedia KB (~21 min) ─────────────────────
# Design: CFR retrieves context sentences from WIKIPEDIA, but gets
# contrastive labels (kb_correct, kb_incorrect) from the dataset-KB.
# This is the fair test: same retrieval corpus as SDCP-Wiki.

print('Building dataset-KB index for TQA...')
tqa_kb_idx = build_kb_index(tqa_kb)

print(f'\n=== CFR | Wikipedia KB | TruthfulQA | test={len(tqa)}Q ===')
alpha = CFR_PARAMS['alpha']
beta  = CFR_PARAMS['beta']
gamma = CFR_PARAMS['gamma']

t0 = time.time()
cfr_tqa_gen, cfr_tqa_ref = [], []

for idx, row in tqdm(tqa.iterrows(), total=len(tqa), desc='CFR-Wiki/TQA'):
    query       = row['question']
    best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]

    # Retrieve context sentences from Wikipedia
    wiki_passages = retrieve_wiki(query, k=20)
    sentences = []
    for p in wiki_passages:
        for s in re.split(r'(?<=[.!?])\s+', p):
            if len(s.strip()) > 15:
                sentences.append(s.strip())

    # Get contrastive labels from dataset-KB
    kb_hits = retrieve_from_kb(query, tqa_kb_idx, tqa_kb, k=1)
    if kb_hits and sentences:
        kb          = kb_hits[0]
        kb_correct  = kb['best_answer'][0] if isinstance(kb['best_answer'],list) else str(kb['best_answer'])
        kb_incorrect= kb['incorrect_answers'][0] if isinstance(kb['incorrect_answers'],list) else ''
        kb_q        = kb.get('question_plain', kb['question'])

        if kb_correct and kb_incorrect:
            s_embs = embed_model.encode(sentences[:100], show_progress_bar=False)
            q_emb  = embed_model.encode([query],         show_progress_bar=False)
            c_emb  = embed_model.encode([kb_correct],    show_progress_bar=False)
            i_emb  = embed_model.encode([kb_incorrect],  show_progress_bar=False)
            scores = (alpha * cosine_similarity(s_embs, q_emb).flatten() +
                      beta  * cosine_similarity(s_embs, c_emb).flatten() -
                      gamma * cosine_similarity(s_embs, i_emb).flatten())
            context = ' '.join([sentences[i] for i in np.argsort(-scores)[:5]])
            prompt  = (f'{INST_S}{SYS}\nContext: {context}\n'
                       f'Example — Q: {kb_q}\nCorrect: {kb_correct}\nIncorrect: {kb_incorrect}\n'
                       f'Question: {query}\nAnswer:{INST_E}')
        else:
            context = ' '.join(sentences[:5])
            prompt  = f'{INST_S}{SYS}\nContext: {context}\nQuestion: {query}\nAnswer:{INST_E}'
    else:
        prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer:{INST_E}'

    cfr_tqa_gen.append(clean_response(generate([prompt])[0]))
    cfr_tqa_ref.append(best_answer)
    if idx % 50 == 0:
        gc.collect(); torch.cuda.empty_cache()

elapsed = (time.time() - t0) / 60
cfr_tqa_metrics = compute_metrics(cfr_tqa_gen, cfr_tqa_ref, tqa)
print(f'\nCFR | TQA | Wikipedia KB | {elapsed:.1f} min')
print(f'  R1={cfr_tqa_metrics["R1"]}  R2={cfr_tqa_metrics["R2"]}  ECS={cfr_tqa_metrics["ECS"]}')

# Save
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
pd.DataFrame({'generated': cfr_tqa_gen,
              'reference': [r[0] if isinstance(r,list) else r for r in cfr_tqa_ref]})\
  .to_csv(f'{OUT_DIR}/cfr_wiki_tqa_{ts}.csv', index=False)
print(f'  Saved: cfr_wiki_tqa_{ts}.csv')

Building dataset-KB index for TQA...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


=== CFR | Wikipedia KB | TruthfulQA | test=615Q ===


CFR-Wiki/TQA: 100%|██████████| 615/615 [1:28:42<00:00,  8.65s/it]



CFR | TQA | Wikipedia KB | 88.7 min
  R1=34.87  R2=20.46  ECS=65.66
  Saved: cfr_wiki_tqa_20260512_131136.csv


In [6]:
# ── Cell 5: CFR on MMLU with Wikipedia KB (~55 min) ───────────────────────────
print('Building dataset-KB index for MMLU...')
mmlu_kb_idx = build_kb_index(mmlu_kb)

print(f'\n=== CFR | Wikipedia KB | MMLU | test={len(mmlu)}Q ===')
t0 = time.time()
cfr_mmlu_gen, cfr_mmlu_ref = [], []

for idx, row in tqdm(mmlu.iterrows(), total=len(mmlu), desc='CFR-Wiki/MMLU'):
    query       = row['question']
    best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]

    wiki_passages = retrieve_wiki(row['question_plain'], k=20)
    sentences = []
    for p in wiki_passages:
        for s in re.split(r'(?<=[.!?])\s+', p):
            if len(s.strip()) > 15:
                sentences.append(s.strip())

    kb_hits = retrieve_from_kb(row['question_plain'], mmlu_kb_idx, mmlu_kb, k=1)
    if kb_hits and sentences:
        kb          = kb_hits[0]
        kb_correct  = kb['best_answer'][0] if isinstance(kb['best_answer'],list) else str(kb['best_answer'])
        kb_incorrect= kb['incorrect_answers'][0] if isinstance(kb['incorrect_answers'],list) else ''
        kb_q        = kb.get('question_plain', kb['question'])

        if kb_correct and kb_incorrect:
            s_embs = embed_model.encode(sentences[:100], show_progress_bar=False)
            q_emb  = embed_model.encode([row['question_plain']], show_progress_bar=False)
            c_emb  = embed_model.encode([kb_correct],            show_progress_bar=False)
            i_emb  = embed_model.encode([kb_incorrect],          show_progress_bar=False)
            scores = (alpha * cosine_similarity(s_embs, q_emb).flatten() +
                      beta  * cosine_similarity(s_embs, c_emb).flatten() -
                      gamma * cosine_similarity(s_embs, i_emb).flatten())
            context = ' '.join([sentences[i] for i in np.argsort(-scores)[:5]])
            prompt  = (f'{INST_S}{SYS}\nContext: {context}\n'
                       f'Example — Q: {kb_q}\nCorrect: {kb_correct}\nIncorrect: {kb_incorrect}\n'
                       f'Question: {query}\nAnswer:{INST_E}')
        else:
            context = ' '.join(sentences[:5])
            prompt  = f'{INST_S}{SYS}\nContext: {context}\nQuestion: {query}\nAnswer:{INST_E}'
    else:
        prompt = f'{INST_S}{SYS}\nQuestion: {query}\nAnswer:{INST_E}'

    cfr_mmlu_gen.append(clean_response(generate([prompt])[0]))
    cfr_mmlu_ref.append(best_answer)
    if idx % 50 == 0:
        gc.collect(); torch.cuda.empty_cache()

elapsed = (time.time() - t0) / 60
cfr_mmlu_metrics = compute_metrics(cfr_mmlu_gen, cfr_mmlu_ref, mmlu)
print(f'\nCFR | MMLU | Wikipedia KB | {elapsed:.1f} min')
print(f'  R1={cfr_mmlu_metrics["R1"]}  Acc={cfr_mmlu_metrics["Acc"]}  ECS={cfr_mmlu_metrics["ECS"]}')

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
pd.DataFrame({'generated': cfr_mmlu_gen,
              'reference': [r[0] if isinstance(r,list) else r for r in cfr_mmlu_ref]})\
  .to_csv(f'{OUT_DIR}/cfr_wiki_mmlu_{ts}.csv', index=False)
print(f'  Saved: cfr_wiki_mmlu_{ts}.csv')

Building dataset-KB index for MMLU...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]


=== CFR | Wikipedia KB | MMLU | test=1596Q ===


CFR-Wiki/MMLU: 100%|██████████| 1596/1596 [4:00:38<00:00,  9.05s/it]  



CFR | MMLU | Wikipedia KB | 240.6 min
  R1=43.94  Acc=None  ECS=56.3
  Saved: cfr_wiki_mmlu_20260512_171217.csv


In [7]:
# ── Cell 6: CAFD-LC — precompute analyses on dataset-KB ───────────────────────
# Pre-compute contrastive fact analyses on the dataset-KB (one-time cost)
print('=== CAFD-LC: Precomputing contrastive analyses on TQA KB ===')
t0 = time.time()
tqa_cafd_cache = {}
for idx, row in tqdm(tqa_kb.iterrows(), total=len(tqa_kb), desc='CAFD-precompute/TQA'):
    correct   = row['best_answer'][0] if isinstance(row['best_answer'],list) else str(row['best_answer'])
    incorrect = row['incorrect_answers'][0] if isinstance(row['incorrect_answers'],list) and row['incorrect_answers'] else ''
    question  = row.get('question_plain', row['question'])
    if not correct or not incorrect:
        tqa_cafd_cache[idx] = None; continue
    prompt = (f'{INST_S}For the question: "{question}"\n'
              f'Correct answer: "{correct}"\n'
              f'Wrong answer: "{incorrect}"\n'
              f'In one sentence explain why correct is right and wrong is misleading:{INST_E}')
    analysis = generate([prompt], max_new_tokens=CAFD_PARAMS['analysis_max_tokens'])[0]
    for stop in ['\n\n','\nQuestion:','\nFor the question','\n---']:
        if stop in analysis: analysis = analysis[:analysis.index(stop)]
    tqa_cafd_cache[idx] = {
        'correct': correct, 'incorrect': incorrect,
        'analysis': analysis.strip()[:200], 'question': question
    }
    if idx % 50 == 0: gc.collect(); torch.cuda.empty_cache()

print(f'  TQA CAFD cache: {sum(1 for v in tqa_cafd_cache.values() if v)} / {len(tqa_kb)} ({(time.time()-t0)/60:.1f} min)')

print('\n=== CAFD-LC: Precomputing contrastive analyses on MMLU KB ===')
t0 = time.time()
mmlu_cafd_cache = {}
for idx, row in tqdm(mmlu_kb.iterrows(), total=len(mmlu_kb), desc='CAFD-precompute/MMLU'):
    correct   = row['best_answer'][0] if isinstance(row['best_answer'],list) else str(row['best_answer'])
    incorrect = row['incorrect_answers'][0] if isinstance(row['incorrect_answers'],list) and row['incorrect_answers'] else ''
    question  = row.get('question_plain', row['question'])
    if not correct or not incorrect:
        mmlu_cafd_cache[idx] = None; continue
    prompt = (f'{INST_S}For the question: "{question}"\n'
              f'Correct answer: "{correct}"\n'
              f'Wrong answer: "{incorrect}"\n'
              f'In one sentence explain why correct is right and wrong is misleading:{INST_E}')
    analysis = generate([prompt], max_new_tokens=CAFD_PARAMS['analysis_max_tokens'])[0]
    for stop in ['\n\n','\nQuestion:','\nFor the question','\n---']:
        if stop in analysis: analysis = analysis[:analysis.index(stop)]
    mmlu_cafd_cache[idx] = {
        'correct': correct, 'incorrect': incorrect,
        'analysis': analysis.strip()[:200], 'question': question
    }
    if idx % 50 == 0: gc.collect(); torch.cuda.empty_cache()

print(f'  MMLU CAFD cache: {sum(1 for v in mmlu_cafd_cache.values() if v)} / {len(mmlu_kb)} ({(time.time()-t0)/60:.1f} min)')

=== CAFD-LC: Precomputing contrastive analyses on TQA KB ===


CAFD-precompute/TQA: 100%|██████████| 100/100 [30:38<00:00, 18.39s/it]


  TQA CAFD cache: 100 / 100 (30.6 min)

=== CAFD-LC: Precomputing contrastive analyses on MMLU KB ===


CAFD-precompute/MMLU: 100%|██████████| 228/228 [1:11:14<00:00, 18.75s/it]

  MMLU CAFD cache: 228 / 228 (71.2 min)


In [8]:
# ── Cell 7: CAFD-LC on TruthfulQA with Wikipedia KB (~25 min) ─────────────────
print(f'\n=== CAFD-LC | Wikipedia KB | TruthfulQA | test={len(tqa)}Q ===')
t0 = time.time()
cafd_tqa_gen, cafd_tqa_ref = [], []

for idx, row in tqdm(tqa.iterrows(), total=len(tqa), desc='CAFD-LC-Wiki/TQA'):
    query       = row['question']
    best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]

    # Context from Wikipedia
    wiki_passages = retrieve_wiki(query, k=5)
    context = ' '.join(wiki_passages[:3])[:800]

    # Contrastive framing from dataset-KB
    kb_hits = retrieve_from_kb(query, tqa_kb_idx, tqa_kb, k=1)
    if kb_hits:
        kb   = kb_hits[0]
        fact = tqa_cafd_cache.get(kb.name)
        if fact:
            kb_q = fact.get('question', kb['question'])
            ctx_block = (f'Context: {context}\n'
                         f'Factual distinction example:\n'
                         f'- Q: {kb_q}\n'
                         f'- FACT: {fact["correct"]}\n'
                         f'- MYTH: {fact["incorrect"]}' +
                         (f'\n- WHY: {fact["analysis"]}' if fact.get('analysis') else ''))
            prompt = f'{INST_S}{SYS}\n{ctx_block}\n---\nQuestion: {query}\nAnswer:{INST_E}'
        else:
            prompt = f'{INST_S}{SYS}\nContext: {context}\nQuestion: {query}\nAnswer:{INST_E}'
    else:
        prompt = f'{INST_S}{SYS}\nContext: {context}\nQuestion: {query}\nAnswer:{INST_E}'

    cafd_tqa_gen.append(clean_response(generate([prompt])[0]))
    cafd_tqa_ref.append(best_answer)
    if idx % 50 == 0:
        gc.collect(); torch.cuda.empty_cache()

elapsed = (time.time() - t0) / 60
cafd_tqa_metrics = compute_metrics(cafd_tqa_gen, cafd_tqa_ref, tqa)
print(f'\nCAFD-LC | TQA | Wikipedia KB | {elapsed:.1f} min')
print(f'  R1={cafd_tqa_metrics["R1"]}  R2={cafd_tqa_metrics["R2"]}  ECS={cafd_tqa_metrics["ECS"]}')

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
pd.DataFrame({'generated': cafd_tqa_gen,
              'reference': [r[0] if isinstance(r,list) else r for r in cafd_tqa_ref]})\
  .to_csv(f'{OUT_DIR}/cafd_wiki_tqa_{ts}.csv', index=False)
print(f'  Saved: cafd_wiki_tqa_{ts}.csv')


=== CAFD-LC | Wikipedia KB | TruthfulQA | test=615Q ===


CAFD-LC-Wiki/TQA: 100%|██████████| 615/615 [1:32:04<00:00,  8.98s/it]



CAFD-LC | TQA | Wikipedia KB | 92.1 min
  R1=34.42  R2=20.08  ECS=65.53
  Saved: cafd_wiki_tqa_20260512_202616.csv


In [9]:
# ── Cell 8: CAFD-LC on MMLU with Wikipedia KB (~60 min) ──────────────────────
print(f'\n=== CAFD-LC | Wikipedia KB | MMLU | test={len(mmlu)}Q ===')
t0 = time.time()
cafd_mmlu_gen, cafd_mmlu_ref = [], []

for idx, row in tqdm(mmlu.iterrows(), total=len(mmlu), desc='CAFD-LC-Wiki/MMLU'):
    query       = row['question']
    best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]

    wiki_passages = retrieve_wiki(row['question_plain'], k=5)
    context = ' '.join(wiki_passages[:3])[:800]

    kb_hits = retrieve_from_kb(row['question_plain'], mmlu_kb_idx, mmlu_kb, k=1)
    if kb_hits:
        kb   = kb_hits[0]
        fact = mmlu_cafd_cache.get(kb.name)
        if fact:
            kb_q = fact.get('question', kb.get('question_plain', kb['question']))
            ctx_block = (f'Context: {context}\n'
                         f'Factual distinction example:\n'
                         f'- Q: {kb_q}\n'
                         f'- FACT: {fact["correct"]}\n'
                         f'- MYTH: {fact["incorrect"]}' +
                         (f'\n- WHY: {fact["analysis"]}' if fact.get('analysis') else ''))
            prompt = f'{INST_S}{SYS}\n{ctx_block}\n---\nQuestion: {query}\nAnswer:{INST_E}'
        else:
            prompt = f'{INST_S}{SYS}\nContext: {context}\nQuestion: {query}\nAnswer:{INST_E}'
    else:
        prompt = f'{INST_S}{SYS}\nContext: {context}\nQuestion: {query}\nAnswer:{INST_E}'

    cafd_mmlu_gen.append(clean_response(generate([prompt])[0]))
    cafd_mmlu_ref.append(best_answer)
    if idx % 50 == 0:
        gc.collect(); torch.cuda.empty_cache()

elapsed = (time.time() - t0) / 60
cafd_mmlu_metrics = compute_metrics(cafd_mmlu_gen, cafd_mmlu_ref, mmlu)
print(f'\nCAFD-LC | MMLU | Wikipedia KB | {elapsed:.1f} min')
print(f'  R1={cafd_mmlu_metrics["R1"]}  Acc={cafd_mmlu_metrics["Acc"]}  ECS={cafd_mmlu_metrics["ECS"]}')

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
pd.DataFrame({'generated': cafd_mmlu_gen,
              'reference': [r[0] if isinstance(r,list) else r for r in cafd_mmlu_ref]})\
  .to_csv(f'{OUT_DIR}/cafd_wiki_mmlu_{ts}.csv', index=False)
print(f'  Saved: cafd_wiki_mmlu_{ts}.csv')


=== CAFD-LC | Wikipedia KB | MMLU | test=1596Q ===


CAFD-LC-Wiki/MMLU: 100%|██████████| 1596/1596 [4:10:01<00:00,  9.40s/it]  



CAFD-LC | MMLU | Wikipedia KB | 250.0 min
  R1=44.48  Acc=None  ECS=56.69
  Saved: cafd_wiki_mmlu_20260513_003620.csv


In [10]:
# ── Cell 9: Final comparison table ───────────────────────────────────────────
print('=' * 72)
print('WIKIPEDIA KB COMPARISON — All methods on same KB')
print('=' * 72)
print(f'\n{"Method":<35} {"TQA R1":>8} {"MMLU R1":>8} {"MMLU Acc":>10}')
print('-' * 65)

# Li et al. baselines on Wikipedia KB
print(f'{"Li et al. Base (Wikipedia)":<35} {26.81:>8} {10.42:>8} {"--":>10}')
print(f'{"Li et al. ICL1Doc+ (Wikipedia)":<35} {30.62:>8} {25.87:>8} {"--":>10}')
print(f'{"Li et al. ICL2Doc+ (Wikipedia)":<35} {30.24:>8} {26.01:>8} {"--":>10}')
print('-' * 65)

# SDCP-v2 on Wikipedia KB (already computed)
print(f'{"SDCP-v2 (Wikipedia KB)":<35} {32.46:>8} {38.40:>8} {"53.3%":>10}')
print('-' * 65)

# New results
print(f'{"CFR (Wikipedia KB) [NEW]":<35} {cfr_tqa_metrics["R1"]:>8} {cfr_mmlu_metrics["R1"]:>8} {str(cfr_mmlu_metrics["Acc"])+"%" if cfr_mmlu_metrics["Acc"] else "--":>10}')
print(f'{"CAFD-LC (Wikipedia KB) [NEW]":<35} {cafd_tqa_metrics["R1"]:>8} {cafd_mmlu_metrics["R1"]:>8} {str(cafd_mmlu_metrics["Acc"])+"%" if cafd_mmlu_metrics["Acc"] else "--":>10}')
print('=' * 72)

# Save summary JSON
ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
out = {
    'timestamp': ts,
    'CFR_wiki_TQA':  cfr_tqa_metrics,
    'CFR_wiki_MMLU': cfr_mmlu_metrics,
    'CAFD_wiki_TQA':  cafd_tqa_metrics,
    'CAFD_wiki_MMLU': cafd_mmlu_metrics,
}
with open(f'{OUT_DIR}/cfr_cafd_wiki_results_{ts}.json', 'w') as f:
    json.dump(out, f, indent=2)
print(f'\nResults saved: cfr_cafd_wiki_results_{ts}.json')
print('\n>>> Next: update Table 1 in emnlp2023.tex with these Wikipedia-KB rows')

WIKIPEDIA KB COMPARISON — All methods on same KB

Method                                TQA R1  MMLU R1   MMLU Acc
-----------------------------------------------------------------
Li et al. Base (Wikipedia)             26.81    10.42         --
Li et al. ICL1Doc+ (Wikipedia)         30.62    25.87         --
Li et al. ICL2Doc+ (Wikipedia)         30.24    26.01         --
-----------------------------------------------------------------
SDCP-v2 (Wikipedia KB)                 32.46     38.4      53.3%
-----------------------------------------------------------------
CFR (Wikipedia KB) [NEW]               34.87    43.94         --
CAFD-LC (Wikipedia KB) [NEW]           34.42    44.48         --

Results saved: cfr_cafd_wiki_results_20260513_003620.json

>>> Next: update Table 1 in emnlp2023.tex with these Wikipedia-KB rows
